# Этап 3, версия 1.4. Проверка роли площади

Проверяем гипотезу на квартирах: помогает ли заполнение отсутствующей площади или лучше обучаться только на объектах с известной площадью? Сравниваем одинаковый алгоритм LightGBM на одной и той же validation. Финальный test не используется.

## Три варианта обучения

A — все квартиры, площадь заполнена по правилам этапа 2, флаг пропуска сохранён. B — только квартиры с исходно известной площадью. C — все квартиры, но площадь и её флаг исключены из признаков. Варианты A и B оцениваются на одних и тех же квартирах с известной площадью; A и C дополнительно сравниваются там, где площадь отсутствует.

In [ ]:
# Загружаем подготовку этапа 2 v1.7 без повторного показа его графиков.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
stage2=next((p/"notebooks"/"02_preprocessing_v1.7.ipynb"
             for p in (Path.cwd(),Path.cwd().parent)
             if (p/"notebooks"/"02_preprocessing_v1.7.ipynb").exists()),None)
if stage2 is None:
    raise FileNotFoundError("Не найден этап 2 v1.7")
with capture_output():
    for cell in json.loads(stage2.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"]=="code":
            result=get_ipython().run_cell("".join(cell["source"]))
            if result.error_in_exec:
                raise result.error_in_exec
import numpy as np
import pandas as pd
from IPython.display import display
ap_train=train.sub_type.eq("Daire")
ap_val=val.sub_type.eq("Daire")
size_train=pd.to_numeric(train["size"],errors="coerce")
size_val=pd.to_numeric(val["size"],errors="coerce")
known_train=ap_train & size_train.ge(15)
known_val=ap_val & size_val.ge(15)
missing_val=ap_val & ~size_val.ge(15)
print("Квартир train:",ap_train.sum(),"из них с известной площадью:",known_train.sum())
print("Квартир validation:",ap_val.sum(),"из них с известной площадью:",known_val.sum(),
      "без достоверной площади:",missing_val.sum())

Известной считаем исходную площадь не меньше 15 м². Проверочные группы формируются до обучения моделей. Заполненная медианой площадь не считается известной.

## Обучение с одинаковым алгоритмом

В каждом варианте числовые пропуски заполняются медианой его train, категории кодируются One-Hot. Настройки LightGBM и логарифм целевой цены совпадают; меняются только строки обучения или использование площади.

In [ ]:
import time
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.metrics import mean_absolute_error,median_absolute_error,r2_score
from lightgbm import LGBMRegressor

def build_model(with_area=True):
    numbers=[c for c in num if with_area or c not in ("size_m2","size_missing")]
    encoder=ColumnTransformer([
        ("num",Pipeline([("fill",SimpleImputer(strategy="median")),
                         ("scale",StandardScaler())]),numbers),
        ("cat",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),
                         ("onehot",OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                 min_frequency=50))]),cat)
    ],sparse_threshold=.8)
    reg=TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42,verbosity=-1,n_jobs=4,num_leaves=15),
        func=np.log1p,inverse_func=np.expm1)
    return Pipeline([("encode",encoder),("model",reg)])

variants={
    "A: все, площадь заполнена":(ap_train,True),
    "B: только известная площадь":(known_train,True),
    "C: без площади":(ap_train,False),
}
models={}
for name,(mask,with_area) in variants.items():
    model=build_model(with_area)
    start=time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train.loc[mask],train.loc[mask,"price"])
    models[name]=model
    print(name,"обучена на",mask.sum(),"строках за",round(time.perf_counter()-start,2),"с")

Во всех трёх вариантах цены validation не участвовали в обучении. Модель B видит меньше примеров, но каждый её тренировочный объект имеет настоящую площадь; A получает больше примеров, часть с приближённой площадью.

In [ ]:
def evaluate(mask):
    y=val.loc[mask,"price"].to_numpy()
    rows=[]; preds={}
    for name,model in models.items():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pred=model.predict(X_val.loc[mask])
        preds[name]=pred
        rows.append({"Модель":name,"Объектов":len(y),
                     "MAE, TRY":mean_absolute_error(y,pred),
                     "MedAE, TRY":median_absolute_error(y,pred),
                     "R²":r2_score(y,pred)})
    return pd.DataFrame(rows),preds,y
all_table,all_predictions,y_all=evaluate(ap_val)
known_table,known_predictions,y_known=evaluate(known_val)
missing_table,missing_predictions,y_missing=evaluate(missing_val)
print("Validation: все квартиры")
display(all_table.round(2))
print("Validation: квартиры с известной площадью")
display(known_table.round(2))
print("Validation: квартиры без достоверной площади")
display(missing_table.round(2))
print("R² без округления на всех квартирах:")
for _,row in all_table.iterrows():
    print(row["Модель"], round(row["R²"],6))


На квартирах с известной площадью A и B сравниваются на одних объектах. Общая таблица показывает цену такого решения: модель, обученная без строк с пропуском, может проиграть на объектах с неизвестной площадью. Вариант C проверяет, полезен ли сам признак площади.


## Насколько устойчива разница

Для A и B на квартирах с известной площадью считаем парную разницу абсолютных ошибок. Повторно выбираем целые группы похожих объявлений, чтобы получить приближённый 95%-й интервал. Положительная разница означает преимущество B.

In [ ]:
a_name="A: все, площадь заполнена"
b_name="B: только известная площадь"
c_name="C: без площади"
delta=np.abs(y_known-known_predictions[a_name])-np.abs(y_known-known_predictions[b_name])
group=pd.util.hash_pandas_object(
    val.loc[known_val,["address","sub_type","room_count","size"]],index=False).to_numpy()
aggregated=pd.DataFrame({"group":group,"delta":delta}).groupby("group").delta.agg(["sum","size"])
sums=aggregated["sum"].to_numpy(); counts=aggregated["size"].to_numpy()
rng=np.random.default_rng(42)
boot=[]
for _ in range(500):
    ids=rng.integers(0,len(aggregated),size=len(aggregated))
    boot.append(sums[ids].sum()/counts[ids].sum())
ci=np.quantile(boot,[.025,.975])
print("Преимущество B по MAE, TRY:",round(delta.mean(),2))
print("95%-й групповой интервал:",[float(round(x,2)) for x in ci])
print("Убедительное преимущество B:",bool(ci[0]>0))
print("Преимущество A над C на известных площадях:",
      round(known_table.loc[known_table.Модель.eq(c_name),"MAE, TRY"].iloc[0]
            -known_table.loc[known_table.Модель.eq(a_name),"MAE, TRY"].iloc[0],2),"TRY")

## Интерпретация проверки площади
Объявление № 392157 исключено на этапе 2 после разбиения и до обучения и оценки. Остальные ID сохранили свои части. На 36 592 квартирах валидации модель A (площадь заполнена) получила MAE 114 686 TRY, MedAE 47 227 TRY и R² 0,093. Модели B и C на той же выборке дали R² 0,082 и 0,089 соответственно. Одна подозрительная цена сильно искажала среднеквадратичную метрику, но удаление этой записи не устраняет остальные аномалии и не означает, что модель хорошо оценивает каждую квартиру.